# Transformer 机器翻译练习版

这个 notebook 用于课堂练习，基于 `MT-transformer-5.ipynb` 改写而来。

本版本只保留 10 个核心 TODO，重点练习机器翻译流程中最重要的部分：

1. 数据路径与读取
2. 分词
3. 词表和 token id
4. padding 与 DataLoader
5. Transformer 关键模块
6. 训练输入/目标错位
7. 推理生成

其余辅助代码已经给出，学生补全 TODO 后即可运行完整流程。


In [ ]:
# pip install torch numpy
# 如果使用 Ascend NPU，需要环境中已安装对应版本的 torch_npu


In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

# 设置随机种子以确保可重复性
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# 指定运行设备：'auto'、'npu'、'cuda'、'mps'、'cpu'
# 在 Ascend 环境中使用 NPU 时，可以改成 DEVICE_NAME = 'npu'
DEVICE_NAME = 'auto'
NPU_ID = 0

def get_device(device_name='auto', npu_id=0):
    if device_name in ('auto', 'npu'):
        try:
            import torch_npu  # noqa: F401
            if hasattr(torch, 'npu') and torch.npu.is_available():
                device = torch.device(f'npu:{npu_id}')
                torch.npu.set_device(f'npu:{npu_id}')
                return device
        except ImportError:
            if device_name == 'npu':
                raise RuntimeError('已指定 DEVICE_NAME="npu"，但当前环境没有安装 torch_npu')
        if device_name == 'npu':
            raise RuntimeError('已指定 DEVICE_NAME="npu"，但 torch.npu.is_available() 不是 True')

    if device_name in ('auto', 'cuda') and torch.cuda.is_available():
        return torch.device('cuda')
    if device_name == 'cuda':
        raise RuntimeError('已指定 DEVICE_NAME="cuda"，但 CUDA 不可用')

    if device_name in ('auto', 'mps') and hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if device_name == 'mps':
        raise RuntimeError('已指定 DEVICE_NAME="mps"，但 MPS 不可用')

    return torch.device('cpu')

device = get_device(DEVICE_NAME, NPU_ID)
if device.type == 'npu':
    torch.npu.manual_seed_all(42)
elif device.type == 'cuda':
    torch.cuda.manual_seed_all(42)
print(f'Using device: {device}')

# TODO 1: 指定数据目录和中英文文件路径
# 提示：DATA_DIR = Path('data')，中文文件是 chinese.txt，英文文件是 english.txt
DATA_DIR = Path('data')
CHINESE_PATH = DATA_DIR / 'chinese.txt'
ENGLISH_PATH = DATA_DIR / 'english.txt'

MAX_SAMPLES = 2000

def load_parallel_corpus(chinese_path, english_path, max_samples=None):
    chinese_lines = chinese_path.read_text(encoding='utf-8').splitlines()
    english_lines = english_path.read_text(encoding='utf-8').splitlines()

    if len(chinese_lines) != len(english_lines):
        raise ValueError(f'中英文行数不一致: {len(chinese_lines)} vs {len(english_lines)}')

    pairs = [
        (chinese.strip(), english.strip())
        for chinese, english in zip(chinese_lines, english_lines)
        if chinese.strip() and english.strip()
    ]
    if max_samples is not None:
        pairs = pairs[:max_samples]

    chinese_sentences = [chinese for chinese, _ in pairs]
    english_sentences = [english for _, english in pairs]
    return chinese_sentences, english_sentences

chinese_sentences, english_sentences = load_parallel_corpus(CHINESE_PATH, ENGLISH_PATH, MAX_SAMPLES)
print(f'Loaded {len(chinese_sentences)} sentence pairs')
print(chinese_sentences[0])
print(english_sentences[0])


Using device: cuda
Loaded 2000 sentence pairs
1998年 , 经过 统一 部署 , 伊犁州 , 地 两 级 党委 开始 尝试 以 宣讲 团 的 形式 , 深入 学校 , 村民 院落 , 田间 地头 , 向 各族 群众 进行 面对面 宣讲 .
in 1998 , the yili autonomous prefecture cpc committee and the yili prefecture cpc committee made unified arrangements and sent on a trial basis several propaganda teams deep into the schools , villagers ' courtyards , and fields to carry out face - to - face propaganda among the people of all nationalities .


In [2]:
# data 目录中的语料已经用空格完成分词，这里直接按空格切分。
def tokenize_ch(text):
    # TODO 2: 按空格切分中英文句子，返回 token 列表
    return text.split()

def tokenize_en(text):
    return text.split()

from collections import Counter

def build_vocab(data, min_freq=1):
    counter = Counter()
    for tokens in data:
        counter.update(tokens)

    # TODO 3: 为普通 token 分配 id，id 从 4 开始
    vocab = {}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab) + 4

    vocab['<pad>'] = 0
    vocab['<sos>'] = 1
    vocab['<eos>'] = 2
    vocab['<unk>'] = 3
    return vocab

chinese_vocab = build_vocab([tokenize_ch(s) for s in chinese_sentences])
english_vocab = build_vocab([tokenize_en(s) for s in english_sentences])

def sentence_to_indices(sentence, vocab):
    # TODO 4: 在句首加入 <sos>，句尾加入 <eos>，未知词使用 <unk>
    sentence = ['<sos>'] + sentence + ['<eos>']
    indices = [vocab.get(token, vocab['<unk>']) for token in sentence]
    return indices

data = [
    (
        sentence_to_indices(tokenize_ch(chinese), chinese_vocab),
        sentence_to_indices(tokenize_en(english), english_vocab),
    )
    for chinese, english in zip(chinese_sentences, english_sentences)
]

print(f'Chinese vocab size: {len(chinese_vocab)}')
print(f'English vocab size: {len(english_vocab)}')
print('First indexed pair:', data[0])


Chinese vocab size: 7795
English vocab size: 6466
First indexed pair: ([1, 4, 5, 6, 7, 8, 5, 9, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 5, 21, 22, 5, 23, 24, 5, 25, 26, 5, 27, 28, 29, 30, 31, 17, 32, 2], [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 7, 8, 10, 11, 12, 14, 15, 16, 13, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 7, 27, 6, 28, 29, 30, 6, 13, 31, 32, 33, 34, 35, 36, 32, 36, 35, 23, 37, 7, 38, 39, 40, 41, 42, 2])


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)

    # TODO 5: 使用 pad_sequence 补齐中文和英文 batch，padding_value=0, batch_first=False
    src_pad = pad_sequence(
        [torch.tensor(seq) for seq in src_batch], 
        padding_value=0, 
        batch_first=False,
    )

    trg_pad = pad_sequence(
        [torch.tensor(seq) for seq in trg_batch], 
        padding_value=0, 
        batch_first=False,
    )

    return src_pad, trg_pad

class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

dataset = TranslationDataset(data)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)


In [4]:
import math

class Transformer(nn.Module):
    def __init__(self, input_dim, output_dim, d_model, nhead, num_encoder_layers, num_decoder_layers, dim_feedforward, dropout):
        super().__init__()

        # TODO 6: 定义源语言 embedding、目标语言 embedding 和输出线性层
        self.src_embedding = nn.Embedding(input_dim, d_model)
        self.trg_embedding = nn.Embedding(output_dim, d_model)

        self.d_model = d_model
        self.transformer = nn.Transformer(
            d_model,
            nhead,
            num_encoder_layers,
            num_decoder_layers,
            dim_feedforward,
            dropout,
            batch_first=False,
        )

        self.fc_out = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(dropout)

    def _generate_positional_encoding(self, seq_len):
        position = torch.arange(seq_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / self.d_model)
        )
        pe = torch.zeros(seq_len, self.d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(1)

    def forward(self, src, trg, trg_mask=None, padding_mask=None):
        src_seq_length, N = src.shape
        trg_seq_length, N = trg.shape

        src_pos = self._generate_positional_encoding(src_seq_length).to(src.device)
        trg_pos = self._generate_positional_encoding(trg_seq_length).to(trg.device)

        src_pos = src_pos.expand(-1, N, -1)
        trg_pos = trg_pos.expand(-1, N, -1)

        # TODO 7: 对 src 和 trg 做 embedding，加位置编码，再 dropout
        src = self.src_embedding(src) * math.sqrt(self.d_model) + src_pos
        trg = self.trg_embedding(trg) * math.sqrt(self.d_model) + trg_pos
        src = self.dropout(src)
        trg = self.dropout(trg)

        output = self.transformer(
            src,
            trg,
            tgt_mask=trg_mask,
            tgt_key_padding_mask=padding_mask,
        )
        prediction = self.fc_out(output)
        return prediction


In [7]:
INPUT_DIM = len(chinese_vocab)
OUTPUT_DIM = len(english_vocab)
D_MODEL = 32
NHEAD = 2
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
DIM_FEEDFORWARD = 32
DROPOUT = 0.05
EPOCHS = 100

model = Transformer(
    INPUT_DIM,
    OUTPUT_DIM,
    D_MODEL,
    NHEAD,
    NUM_ENCODER_LAYERS,
    NUM_DECODER_LAYERS,
    DIM_FEEDFORWARD,
    DROPOUT,
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    for i, (src, trg) in enumerate(dataloader):
        src = src.to(device)
        trg = trg.to(device)

        # TODO 8: 构造 trg_input 和 trg_expected
        # 提示：trg_input 去掉最后一个 token，trg_expected 去掉第一个 token
        trg_input = trg[:-1, :]
        trg_expected = trg[1:, :]

        trg_mask = nn.Transformer.generate_square_subsequent_mask(trg_input.size(0)).to(device).bool()
        padding_mask = (trg_input == 0).transpose(0, 1)

        # TODO 9: 前向传播并计算交叉熵损失
        output = model(src, trg_input, trg_mask=trg_mask, padding_mask=padding_mask)

        # 提示：output reshape 成 [-1, OUTPUT_DIM]，trg_expected reshape 成 [-1]
        loss = criterion(output.reshape(-1, OUTPUT_DIM), trg_expected.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch {epoch + 1}, Loss: {loss.item()}')


c:\Users\zyh07\Code\Project\AI_Programming\.venv\Lib\site-packages\torch\nn\modules\transformer.py:143: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


Epoch 1, Loss: 6.2877583503723145
Epoch 2, Loss: 5.946749210357666
Epoch 3, Loss: 5.660555362701416
Epoch 4, Loss: 5.7112603187561035
Epoch 5, Loss: 5.500307083129883
Epoch 6, Loss: 5.701159954071045
Epoch 7, Loss: 5.573364734649658
Epoch 8, Loss: 5.249906063079834
Epoch 9, Loss: 5.072826862335205
Epoch 10, Loss: 5.041363716125488
Epoch 11, Loss: 4.73268461227417
Epoch 12, Loss: 4.966233253479004
Epoch 13, Loss: 4.383386135101318
Epoch 14, Loss: 4.548693656921387
Epoch 15, Loss: 4.64151668548584
Epoch 16, Loss: 4.386247158050537
Epoch 17, Loss: 4.029598712921143
Epoch 18, Loss: 4.94455623626709
Epoch 19, Loss: 4.4457316398620605
Epoch 20, Loss: 4.2083892822265625
Epoch 21, Loss: 4.227838039398193
Epoch 22, Loss: 4.008642673492432
Epoch 23, Loss: 3.934481382369995
Epoch 24, Loss: 3.9859650135040283
Epoch 25, Loss: 3.677431583404541
Epoch 26, Loss: 3.8018219470977783
Epoch 27, Loss: 4.136602878570557
Epoch 28, Loss: 3.876607894897461
Epoch 29, Loss: 4.127817153930664
Epoch 30, Loss: 3.88

In [11]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device=None, max_len=50):
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    tokens = tokenize_ch(sentence)
    indices = sentence_to_indices(tokens, src_vocab)
    src_tensor = torch.tensor(indices, dtype=torch.long, device=device).unsqueeze(1)
    trg_indices = [trg_vocab['<sos>']]

    for i in range(max_len):
        trg_tensor = torch.tensor(trg_indices, dtype=torch.long, device=device).unsqueeze(1)
        with torch.no_grad():
            output = model(src_tensor, trg_tensor)

        # TODO 10: 取最后一个时间步概率最大的 token id，加入 trg_indices；生成 <eos> 时停止
        pred_token = output.argmax(dim=-1)[-1, 0].item()
        trg_indices.append(pred_token)

        # 补全停止条件
        if pred_token == trg_vocab['<eos>']:
            break

    idx_to_trg = {idx: token for token, idx in trg_vocab.items()}
    trg_tokens = [idx_to_trg.get(i, '<unk>') for i in trg_indices]
    if trg_tokens and trg_tokens[0] == '<sos>':
        trg_tokens = trg_tokens[1:]
    if '<eos>' in trg_tokens:
        trg_tokens = trg_tokens[:trg_tokens.index('<eos>')]
    return ' '.join(trg_tokens)

test_sentences = chinese_sentences[:4]
for sentence in test_sentences:
    translation = translate_sentence(sentence, chinese_vocab, english_vocab, model, device=device)
    print(f'Translated sentence: {translation}')


Translated sentence: the yili prefecture cpc committee and japanese propaganda teams deep down - year , villagers we will be used to village , and e - chongqing .
Translated sentence: in the grassland , the trees and cadres , and conscientiously listened to conscientiously listened to conscientiously listened to tackle supply and sometimes , and sometimes , and sometimes , and sometimes , cut problems .
Translated sentence: he said : the yining caught the staff guo boxiong is not asked for the ancestral stresses " education is a dragon , jokingly said that the hospital near for the ancestral stresses " education is a lecture to go to go to go method delaying the hospital did not
Translated sentence: yibulayin , however , the candidates sold high - known to the " three stresses " education , it is just go connection with the " education , and the " education is just go connection with the following the following the following stages organs , and they had not
